# DX 704 Week 7 Project

This week's project will investigate issues in a quadcopter controller based using a linear quadratic regulator.
You will start with an existing model of the system and logs from a quadcopter based on it, investigate discrepancies, and ultimately train a new model of the system dynamics.

The full project description and a template notebook are available on GitHub: [Project 7 Materials](https://github.com/bu-cds-dx704/dx704-project-07).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Introduction

You've just joined a drone startup.
On your first day, you join your new team to watch a test flight for a new quadcopter prototype.
Watching the prototype fly, the team comments that it is not as smooth as usual and suspects that something is off in the controller.
They provide you a copy of the dynamics model and log data from the prototype to investigate.

The quadcopter control model is a slightly more sophisticated version of the 1D quadcopter problem previously considered.

The state vector $\mathbf{x}_t$ now includes an acceleration component, and the action vector now has a servo control for the throttle instead of a raw force component.
\begin{array}{rcl}
\mathbf{x}_t & = & \begin{bmatrix} x_t \\ v_t \\ a_t \end{bmatrix} \\
\mathbf{u_t} & = & \begin{bmatrix} u_t \end{bmatrix}
\end{array}

## Part 1: Reconstruct the Control Matrix

You are provided the dynamics model in the files `model-A.tsv`, `model-B.tsv`, `cost-Q.tsv` and `cost-R.tsv`.
Recompute the control matrix $\mathbf{K}$ to be used in the infinite horizon linear quadratic regulator problem.

In [12]:
# YOUR CHANGES HERE
import pandas as pd
import numpy as np
from scipy.linalg import solve_discrete_are

#load matrices from tsv files given
A = pd.read_csv("model-A.tsv", sep='\t', index_col=0).to_numpy() #as np array for matrix math/calculations
B = pd.read_csv("model-B.tsv", sep='\t', index_col=0).to_numpy()
Q = pd.read_csv("cost-Q.tsv", sep='\t', index_col=0).to_numpy()
R = pd.read_csv("cost-R.tsv", sep='\t', index_col=0).to_numpy()

#print matrices
print("A matrix:\n", A)
print("B matrix:\n", B)
print("Q matrix:\n", Q)
print("R matrix:\n", R)

A matrix:
 [[1 1 0]
 [0 1 1]
 [0 0 1]]
B matrix:
 [[0]
 [0]
 [1]]
Q matrix:
 [[5 0 0]
 [0 1 0]
 [0 0 2]]
R matrix:
 [[5]]


In [13]:
#solve riccati eqn
P = solve_discrete_are(A, B, Q, R)
print("Riccati Equation Solution P:\n", P)

Riccati Equation Solution P:
 [[19.50090752 27.77808565 14.94947739]
 [27.77808565 65.61201283 43.35619783]
 [14.94947739 43.35619783 39.69737486]]


In [14]:
#calculate LQR gain
K = np.linalg.inv(R + B.T @ P @ B) @ (B.T @ P @ A)
print("LQR gain K:\n", K)

LQR gain K:
 [[0.33445985 1.30445413 1.85813088]]


Save $\mathbf{K}$ in a file "control-K-intended.tsv" with columns x, v and a.

In [15]:
# YOUR CHANGES HERE
K_df = pd.DataFrame(K, columns=['x', 'v', 'a'])
K_df.to_csv("control-K-intended.tsv", sep='\t', index=False)

Submit "control-K-intended.tsv" in Gradescope.

## Part 2: Recompute the Actions for the Logged States

You get access to the log data for the prototype as the file "qc-log.tsv".
It conveniently saves all the state and actions made.
Recompute the actions based on your $\mathbf{K}$ matrix computed in part 1.

In [ ]:
# YOUR CHANGES HERE
#load qc-log.tsv
qc_log_df = pd.read_csv("qc-log.tsv", sep='\t')
qc_log_df = qc_log_df.drop(columns=['index'], axis=0) #drop index columns
qc_log_df.head(5)

,x,v,a,u
0,-5.000000,0.000000,0.000000,1.702188
1,-5.000000,-0.017022,1.531969,-1.263200
2,-5.018724,1.452683,0.548285,-1.249321
3,-3.420773,1.840779,-0.521275,-0.212127
4,-1.395916,1.163611,-0.764317,0.452895


In [27]:
#extract state cols as 2D np.array (u = original action)
X = qc_log_df[['x', 'v', 'a']].to_numpy()

#LQR policy: u_t = -K * x_t
U_new = -X @ K.T #matrix mult
#add actions to df
qc_log_df['u_new'] = U_new
qc_log_df.head(5)

,x,v,a,u,u_new
0,-5.000000,0.000000,0.000000,1.702188,1.672299
1,-5.000000,-0.017022,1.531969,-1.263200,-1.152095
2,-5.018724,1.452683,0.548285,-1.249321,-1.235183
3,-3.420773,1.840779,-0.521275,-0.212127,-0.288504
4,-1.395916,1.163611,-0.764317,0.452895,0.369202


Save your computed actions as "qc-check.tsv" with columns "index" and "u_check".

In [28]:
# YOUR CHANGES HERE
qc_check_df = pd.DataFrame({
    "index": qc_log_df.index,
    "u_check":  qc_log_df['u_new']
})
qc_check_df.to_csv("qc-check.tsv", sep='\t', index=False)


Submit "qc-check.tsv" in Gradescope.

## Part 3: Reverse Engineer the Actual Control Matrix

Now that you have found a systematic difference between your computed actions and the logged actions, estimate $
$, the control matrix that was actually used to choose actions in the prototype.

Hint: With a linear quadratic regulator, the optimal actions are always linear combinations of the state that are calculated using the control matrix.
You can use linear regression to reverse-engineer the coefficients in the control matrix.

In [31]:
# YOUR CHANGES HERE
from sklearn.linear_model import LinearRegression
#get logged actions from qc-log
u_log = qc_log_df['u'].to_numpy()

#fit lin. reg.
model = LinearRegression(fit_intercept=False)
model.fit(X, u_log)
#extract coefs.
K_actual = model.coef_
K_actual

array([-0.34043755, -1.30012023, -1.95011696])

Save $\mathbf{K}_{\mathrm{actual}}$ in "control-K-actual.tsv" with the same format as "control-K-intended.tsv".

In [38]:
# YOUR CHANGES HERE
#save in df to convert to tsv
control_K_actual_df = pd.DataFrame([K_actual], columns=['x', 'v', 'a'])
control_K_actual_df.to_csv("control-K-actual.tsv", sep='\t', index=False)

Submit "control-k-actual.tsv" in Gradescope.

## Part 4: Recompute the System Dynamics from the Log Data

On further investigation, it turns out that the control matrix $\mathbf{K}$ in the prototype was modified to compensate for state prediction errors.
You would like to recompute the $\mathbf{A}$ and $\mathbf{B}$ matrices using the log data since they are used to predict the next states.
However, since the action vector $\mathbf{u}_t$ is linearly dependent on the state via $\mathbf{u}_t=-\mathbf{K} \mathbf{x}_t$, you need a new data set so you can separate the effects of the $\mathbf{A}$ and $\mathbf{B}$ matrices.
Your co-workers kindly provide a new file "qc-train.tsv" where noise was added to each action.
Estimate the true $\mathbf{A}$ and $\mathbf{B}$ matrices based on this file.

In [42]:
# YOUR CHANGES HERE
#load trianing data
train = pd.read_csv('qc-train.tsv', sep='\t')
train = train.drop(columns=['index'], axis=0) #drop index columns

#extract current states
X = train[['x', 'v', 'a']].to_numpy()

#extract actions
U = train['u'].to_numpy().reshape(-1,1)

#get next states
X_next = train[['x', 'v', 'a']].shift(-1).dropna().to_numpy()
X = X[:-1, :] #drop last row to macth
U = U[:-1, :]

#concatenate X and U for reg. 
XU = np.hstack([X, U])

#solve lin. sys. using least squares: X_next = XU @ [A | B].T
AB_combined, residuals, rank, s = np.linalg.lstsq(XU, X_next, rcond=None)

#combined shape is (4,3) since ;stsq solves XU @ AB_combined = X_next
#want A = first 3 rows, B = last row
A_est = AB_combined[:3, :].T
B_est = AB_combined[3, :].reshape(3,1)

print("Estimated A matrix:\n", A_est)
print("Estimated B matrix:\n", B_est)

Estimated A matrix:
 [[ 1.00000000e+00  1.10000000e+00  2.88657986e-15]
 [-1.28195498e-16  9.00000000e-01  9.50000000e-01]
 [-9.31378031e-17  5.55111512e-16  1.10000000e+00]]
Estimated B matrix:
 [[-1.11022302e-16]
 [-1.00000000e-02]
 [ 9.00000000e-01]]


Save $\mathbf{A}$ and $\mathbf{B}$ to "model-A-new.tsv" and "model-B-new.tsv" respectively.

In [43]:
#save in dfs
A_df = pd.DataFrame(A_est, columns=['x','v','a'])
A_df

,x,v,a
0,1.000000e+00,1.100000e+00,2.886580e-15
1,-1.281955e-16,9.000000e-01,9.500000e-01
2,-9.313780e-17,5.551115e-16,1.100000e+00


In [44]:
B_df = pd.DataFrame(B_est, columns=['u'])
B_df

,u
0,-1.110223e-16
1,-1.000000e-02
2,9.000000e-01


In [45]:
# YOUR CHANGES HERE
A_df.to_csv("model-A-new.tsv", sep='\t', index=False)
B_df.to_csv("model-B-new.tsv", sep='\t', index=False)

Submit "model-A-new.tsv" and "model-B-new.tsv" in Gradescope.

## Part 5: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 6: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

In [46]:
file_content = "For this assignment, I referenced scipy's documentation on solve_discrete_are: https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.solve_discrete_are.html "
with open('acknowledgments.txt', 'w') as f:
    f.write(file_content)